## Importing Libraries

In [1]:
import os
from dotenv import load_dotenv
from graphdatascience import GraphDataScience
import pandas as pd

c:\who-eats-whom\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load and retreive neo4j variables from .env file

In [ ]:
#Storing Neo4j connection details in variables
URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = ""
DB_NAME = ""

## Initialize the Graph Data Science client

In [3]:
gds = GraphDataScience(
    URI,
    auth=(USER, PASSWORD),
    database=DB_NAME
)

In [4]:

print("Connected to Neo4j GDS server version:", gds.version())

Connected to Neo4j GDS server version: 2.25.0


In [5]:
print("Available GDS algorithms (first few rows):")
print(gds.list().head())

Available GDS algorithms (first few rows):
                                         name  \
0           gds.allShortestPaths.delta.mutate   
1  gds.allShortestPaths.delta.mutate.estimate   
2            gds.allShortestPaths.delta.stats   
3   gds.allShortestPaths.delta.stats.estimate   
4           gds.allShortestPaths.delta.stream   

                                         description  \
0  The Delta Stepping shortest path algorithm com...   
1  Returns an estimation of the memory consumptio...   
2  The Delta Stepping shortest path algorithm com...   
3  Returns an estimation of the memory consumptio...   
4  The Delta Stepping shortest path algorithm com...   

                                           signature       type  
0  gds.allShortestPaths.delta.mutate(graphName ::...  procedure  
1  gds.allShortestPaths.delta.mutate.estimate(gra...  procedure  
2  gds.allShortestPaths.delta.stats(graphName :: ...  procedure  
3  gds.allShortestPaths.delta.stats.estimate(grap...  procedu

## Graph Projections

In [6]:
# Cleanup existing graph projection if it exists
if gds.graph.exists("foodweb_directed").exists:
    gds.graph.drop("foodweb_directed")

In [7]:
# First projection for directed analyses (degree, betweenness, closeness, communities)
G, proj_result = gds.graph.project(
    "foodweb_directed",
    {
        "Species": {
            "properties": []
        }
    },
    {
        "eaten_by": {
            "orientation": "NATURAL",   
            "properties": []
        }
    }
)


In [8]:
G_und, proj_result = gds.graph.project(
    "foodweb_undirected",
    {
        "Species": {
            "properties": []
        }
    },
    {
        "eaten_by": {
            "orientation": "UNDIRECTED",   
            "properties": []
        }
    }
)

In [9]:
print(f"Projected {G.node_count()} nodes and {G.relationship_count()} relationships for directed analyses.\n")

Projected 1876 nodes and 1658 relationships for directed analyses.



In [11]:
print(f"Projected {G_und.node_count()} nodes and {G_und.relationship_count()} relationships for undirected analyses.\n")

Projected 1876 nodes and 3316 relationships for undirected analyses.



## Phase 1: Analysis



In [19]:
props_df = gds.run_cypher("""
MATCH ()-[r:eaten_by]->()
UNWIND keys(r) AS k
RETURN k AS property, count(*) AS freq
ORDER BY freq DESC
""")
print(props_df)

                    property  freq
0   predator_scientific_name  1658
1       prey_scientific_name  1658
2                observed_on  1658
3                  time_zone  1658
4                  image_url  1658
5                        url  1658
6         captive_cultivated  1658
7            prey_agreements  1658
8        predator_agreements  1658
9                   latitude  1657
10                 longitude  1657
11               place_state  1655
12             place_country  1655
13                     place  1652
14              place_county  1638
15       positional_accuracy  1401
16           type_of_feeding  1142
17                place_town   121


### Calculating Trophic Levels

#### Find all species nodes that have indegree 0

In [32]:
query = """CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
RETURN gds.util.asNode(nodeId) AS producer
ORDER BY producer.name; """

result = gds.run_cypher(query)
print(result)

                                              producer
0    (iconic_taxon_name, taxon_subfamily, taxon_kin...
1    (iconic_taxon_name, taxon_subfamily, taxon_kin...
2    (iconic_taxon_name, taxon_kingdom, taxon_super...
3    (iconic_taxon_name, taxon_subfamily, taxon_kin...
4    (iconic_taxon_name, taxon_subfamily, taxon_kin...
..                                                 ...
878  (iconic_taxon_name, taxon_kingdom, taxon_subph...
879  (iconic_taxon_name, taxon_kingdom, taxon_super...
880  (iconic_taxon_name, taxon_subfamily, taxon_kin...
881  (iconic_taxon_name, taxon_subfamily, taxon_kin...
882  (iconic_taxon_name, taxon_subfamily, taxon_kin...

[883 rows x 1 columns]


#### Check if all the producers belong to plantae kingdom

In [33]:
query = """CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
RETURN
  producer.taxon_kingdom AS taxon_kingdom_name,
  count(*) AS producer_count
ORDER BY producer_count DESC;"""

result = gds.run_cypher(query)
print(result)

  taxon_kingdom_name  producer_count
0           Animalia             479
1            Plantae             398
2              Fungi               5
3          Chromista               1


#### Find the Phylum name for species from Animal Kingdom

In [34]:
query = """CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom = 'Animalia'
RETURN
  producer.taxon_phylum AS taxon_phylum,
  count(*) AS producer_count
ORDER BY producer_count DESC;"""
result = gds.run_cypher(query)
print(result)

    taxon_phylum  producer_count
0       Chordata             258
1     Arthropoda             196
2       Mollusca              14
3  Echinodermata               6
4       Annelida               2
5       Porifera               1
6    Onychophora               1
7       Cnidaria               1


#### Check unique kingdom names in the database

In [35]:
query = """MATCH (s:Species)
RETURN s.taxon_kingdom AS taxon_kingdom, count(*) AS count
ORDER BY count DESC;"""

result = gds.run_cypher(query)
print(result)

  taxon_kingdom  count
0      Animalia   1436
1       Plantae    417
2         Fungi     22
3     Chromista      1


#### Generate a CSV for verificiation for Animila Kingdom

In [36]:
query = """
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom = 'Animalia'
RETURN
  producer.taxon_kingdom AS kingdom_name,
  producer.taxon_phylum AS taxon_phylum,
  producer.scientific_name as scientific_name
ORDER BY scientific_name, taxon_phylum;"""

result = gds.run_cypher(query)
print(result)

    kingdom_name taxon_phylum               scientific_name
0       Animalia     Chordata                 Abramis brama
1       Animalia   Arthropoda             Acanalonia conica
2       Animalia   Arthropoda     Acanthocephala terminalis
3       Animalia     Chordata      Acanthocercus atricollis
4       Animalia     Chordata      Acanthogobius flavimanus
..           ...          ...                           ...
474     Animalia   Arthropoda  Xylocopa virginica virginica
475     Animalia   Arthropoda           Xystocheir dissecta
476     Animalia     Chordata              Zamenis scalaris
477     Animalia     Chordata              Zenaida macroura
478     Animalia   Arthropoda    Zootermopsis angusticollis

[479 rows x 3 columns]


#### Verify, if all the Plantae phylums are producers

In [38]:
query = """CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom = 'Plantae'
RETURN
  producer.taxon_phylum AS taxon_phylum,
  count(*) AS producer_count
ORDER BY producer_count DESC;  """

result = gds.run_cypher(query)
print(result)


   taxon_phylum  producer_count
0  Tracheophyta             398


#### Set trophic level for plantae and chromista as 1

In [40]:
query = """CALL gds.degree.stream('foodweb_directed', { orientation: 'REVERSE' })
YIELD nodeId, score AS indegree
WITH gds.util.asNode(nodeId) AS producer
WHERE indegree = 0 AND producer.taxon_kingdom IN ['Plantae', 'Chromista'] 
MATCH (producer)
SET producer.trophicLevel = 1.0;"""

gds.run_cypher(query)


""


### Trophic Level 2 - Primary Consumers

In [58]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ['Plantae', 'Chromista']

// Step 2: Find species that eat these producers
MATCH (producer)-[:eaten_by]->(consumer:Species)
RETURN DISTINCT consumer.scientific_name AS consumer_name,
                consumer.common_name AS common_name,
                consumer.taxon_class AS class,
                consumer.taxon_phylum AS phylum
ORDER BY consumer_name;"""

result = gds.run_cypher(query)
print(result)

                 consumer_name                 common_name      class  \
0            Abantis paradisea            Paradise Skipper    Insecta   
1        Abisares viridipennis  Notched Shield Grasshopper    Insecta   
2              Acalitus mallyi       Mispel Leaf Gall Mite  Arachnida   
3              Aceria camdeboo                        None  Arachnida   
4              Aceria lantanae     Lantana Flower Gallmite  Arachnida   
..                         ...                         ...        ...   
466          Zonocerus elegans                        None    Insecta   
467  Zonocerus elegans elegans                        None    Insecta   
468           Zosterops virens        Green Cape White-eye       Aves   
469    Zosterops virens virens        Green Cape White-eye       Aves   
470       Zygaena filipendulae             Six-spot Burnet    Insecta   

         phylum  
0    Arthropoda  
1    Arthropoda  
2    Arthropoda  
3    Arthropoda  
4    Arthropoda  
..          ...

#### Finding Phylum of these Primary Consumers

In [61]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ['Plantae', 'Chromista']

// Step 2: Find species that eat these producers
MATCH (producer)-[:eaten_by]->(consumer:Species)
RETURN DISTINCT consumer.taxon_phylum AS phylum,
                count (DISTINCT consumer) as count
ORDER BY phylum;

"""

result = gds.run_cypher(query)
print(result)

          phylum  count
0     Arthropoda    325
1     Ascomycota      8
2  Basidiomycota      6
3       Chordata    122
4       Mollusca      4
5   Tracheophyta      6


#### Finding Texon Class of Primary Consumers

In [18]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ['Plantae', 'Chromista']

// Step 2: Find species that eat these producers
MATCH (producer)-[:eaten_by]->(consumer:Species)
RETURN DISTINCT consumer.taxon_class AS class,
                count (DISTINCT consumer) as count
ORDER BY class;"""

result = gds.run_cypher(query)
print(result)

                class  count
0      Agaricomycetes      1
1           Arachnida      7
2                Aves     92
3     Dothideomycetes      5
4   Exobasidiomycetes      2
5          Gastropoda      4
6             Insecta    318
7       Leotiomycetes      3
8       Magnoliopsida      6
9            Mammalia     29
10    Pucciniomycetes      3
11           Reptilia      1


In [20]:
result.to_csv('verification/primary_consumers_by_class.csv', index=False)

#### Trophic Level 3 - Secondary Consumers

In [60]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ['Plantae', 'Chromista']

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

// Step 3: Find secondary consumers
MATCH (primary)-[:eaten_by]->(secondary:Species)

// Step 4: Return results
RETURN DISTINCT secondary.scientific_name AS secondary_consumer_name,
                secondary.common_name AS common_name,
                secondary.taxon_class AS class,
                secondary.taxon_phylum AS phylum
ORDER BY secondary_consumer_name;
"""
result = gds.run_cypher(query)
print(result)


       secondary_consumer_name                 common_name      class  \
0            Aerospiza tachiro             African Goshawk       Aves   
1   Alligator mississippiensis          American Alligator   Reptilia   
2           Amblyomma hebraeum     South African Bont Tick  Arachnida   
3        Andrenosoma hesperium   Golden-horned Chiselmouth    Insecta   
4       Apiomerus californicus     California Bee Assassin    Insecta   
..                         ...                         ...        ...   
62        Vespula pensylvanica        Western Yellowjacket    Insecta   
63               Vulpes vulpes                     Red Fox   Mammalia   
64              Zelus longipes       Milkweed Assassin Bug    Insecta   
65            Zodarion rubidum  European Ant-eating Spider  Arachnida   
66           Zygiella x-notata  Silver-sided Sector Spider  Arachnida   

        phylum  
0     Chordata  
1     Chordata  
2   Arthropoda  
3   Arthropoda  
4   Arthropoda  
..         ...  
62  

#### Finding Taxon Phylum of these Secondary Consumers

In [21]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ['Plantae', 'Chromista']

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

// Step 3: Find secondary consumers
MATCH (primary)-[:eaten_by]->(secondary:Species)

// Step 4: Group by phylum and count unique secondary consumers
RETURN secondary.taxon_phylum AS phylum,
       count(DISTINCT secondary) AS count
ORDER BY phylum;"""
result = gds.run_cypher(query)
print(result)

         phylum  count
0    Arthropoda     32
1    Ascomycota      1
2      Chordata     29
3  Tracheophyta      5


#### Finding Taxon Class of these Secondary Consumers

In [22]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ['Plantae', 'Chromista']

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

// Step 3: Find secondary consumers
MATCH (primary)-[:eaten_by]->(secondary:Species)

// Step 4: Group by phylum and count unique secondary consumers
RETURN secondary.taxon_class AS class,
       count(DISTINCT secondary) AS count
ORDER BY class;"""
result = gds.run_cypher(query)
print(result)

             class  count
0        Arachnida     12
1             Aves     17
2          Insecta     20
3    Magnoliopsida      5
4         Mammalia      6
5         Reptilia      6
6  Sordariomycetes      1


In [23]:
result.to_csv("verification/secondary_consumers_by_class.csv", index=False   )

#### Trophic Level 4 - Tertiary Consumers

In [24]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'foodweb_directed',
  { orientation: 'REVERSE' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ['Plantae', 'Chromista']

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

// Step 3: Find secondary consumers
MATCH (primary)-[:eaten_by]->(secondary:Species)

// Step 4: Find tertiary consumers
MATCH (secondary)-[:eaten_by]->(tertiary:Species)

// Step 5: Return results
RETURN DISTINCT
       tertiary.scientific_name AS tertiary_consumer_name,
       tertiary.common_name     AS common_name,
       tertiary.taxon_class     AS class,
       tertiary.taxon_phylum    AS phylum
ORDER BY tertiary_consumer_name;
"""
result = gds.run_cypher(query)
print(result)

         tertiary_consumer_name                          common_name  \
0                   Afrogegenes                              Dodgers   
1         Andrenosoma hesperium            Golden-horned Chiselmouth   
2        Apiomerus californicus              California Bee Assassin   
3                Apis mellifera                    Western Honey Bee   
4              Aquarius remigis  North American Common Water Strider   
5           Argiope trifasciata                 Banded Garden Spider   
6      Belenois creona severina                 African Common White   
7               Bombus bifarius    Colorado Black-notched Bumble Bee   
8           Bombus griseocollis              Brown-belted Bumble Bee   
9             Bombus nevadensis                    Nevada Bumble Bee   
10            Buteo jamaicensis                      Red-tailed Hawk   
11               Cathartes aura                       Turkey Vulture   
12          Chromolaena odorata                            Siam 

In [25]:
result.to_csv("verification/tertiary_consumers.csv", index=False)

### Find number of Decomposers - Fungi

In [41]:
query = """
MATCH (s:Species)
WHERE s.taxon_kingdom = 'Fungi'
RETURN count(s) AS fungi_species_count;"""

result = gds.run_cypher(query)
print(result)

   fungi_species_count
0                   22


### Find Omnivores

In [12]:
query = """MATCH (prey:Species)-[:eaten_by]->(predator:Species)
WITH predator, collect(DISTINCT prey.taxon_kingdom) AS prey_kingdoms
WHERE size(prey_kingdoms) >1
RETURN predator.scientific_name AS omnivore, prey_kingdoms
ORDER BY predator.name;"""

result = gds.run_cypher(query)
print(result)

                       omnivore               prey_kingdoms
0                Apis mellifera         [Animalia, Plantae]
1                 Canis latrans         [Animalia, Plantae]
2              Coragyps atratus         [Animalia, Plantae]
3    Diabrotica undecimpunctata         [Animalia, Plantae]
4           Gallinula chloropus         [Animalia, Plantae]
5            Larus delawarensis         [Animalia, Plantae]
6              Lybius torquatus         [Animalia, Plantae]
7    Lybius torquatus torquatus         [Animalia, Plantae]
8                 Pica hudsonia         [Animalia, Plantae]
9           Pycnonotus barbatus         [Animalia, Plantae]
10  Pycnonotus barbatus layardi         [Animalia, Plantae]
11         Sciurus carolinensis  [Animalia, Plantae, Fungi]
12      Selasphorus platycercus         [Animalia, Plantae]
13          Sylvilagus bachmani         [Animalia, Plantae]
14      Tamiasciurus hudsonicus         [Plantae, Animalia]
15        Tetramorium immigrans         

In [13]:
result.to_csv("verification/omnivores.csv", index=False)

### Finding Decomposers

In [49]:
query = """MATCH (s:Species)
WHERE s.taxon_kingdom = 'Fungi'
RETURN s.scientific_name AS fungi_species_name;"""

result = gds.run_cypher(query)
print(result)

                      fungi_species_name
0                     Allodus podophylli
1                Arthrophaga myriapodina
2                   Ascocoryne sarcoides
3                    Caulorhiza umbonata
4                        Cercospora atra
5                Coniodictyum chevalieri
6                    Coprinopsis lagopus
7                    Cystotheca mexicana
8                Exobasidium rhododendri
9                       Furia ithacensis
10                         Golovinomyces
11  Gymnosporangium juniperi-virginianae
12               Laetiporus gilbertsonii
13                  Macrosporium asimini
14                     Mikronegeria fagi
15              Mycocentrospora asiminae
16              Ophiocordyceps humbertii
17                Phylloporia amplectens
18                 Phyllosticta asiminae
19           Pseudocercospora fuliginosa
20                     Rhytisma acerinum
21                Stereum ochraceoflavum


#### Check if there exist any bacteria in Animalia Kingdom

In [50]:
query = """
MATCH (s:Species)
WHERE s.taxon_kingdom = 'Animalia'
  AND s.taxon_phylum IN [
    'Proteobacteria',
    'Actinobacteria',
    'Firmicutes',
    'Bacteroidota'
  ]
RETURN s.scientific_name, s.common_name;"""

result = gds.run_cypher(query)
print(result)

Empty DataFrame
Columns: [s.scientific_name, s.common_name]
Index: []


#### Finding Decomposers from Animalia Kingdom

In [14]:
query = """MATCH (s:Species)
WHERE s.taxon_kingdom = 'Animalia'
  AND s.taxon_phylum IN ['Annelida', 'Arthropoda', 'Nematoda', 'Mollusca']
RETURN DISTINCT s.scientific_name as scientific_name, s.common_name AS common_name, s.taxon_phylum as phylum_name
ORDER BY phylum_name, scientific_name;"""

result = gds.run_cypher(query)
print(result)

           scientific_name                 common_name phylum_name
0          Alitta williami                        None    Annelida
1       Hirudo michaelseni        Common African Leech    Annelida
2            Urechis caupo          Fat Innkeeper Worm    Annelida
3        Abantis paradisea            Paradise Skipper  Arthropoda
4    Abisares viridipennis  Notched Shield Grasshopper  Arthropoda
..                     ...                         ...         ...
725       Rumina decollata             Decollate Snail    Mollusca
726     Saxidomus gigantea                 Butter Clam    Mollusca
727      Tandonia sowerbyi                 Keeled Slug    Mollusca
728      Testacella maugei                Mauge's Slug    Mollusca
729  Triplofusus giganteus         Florida Horse Conch    Mollusca

[730 rows x 3 columns]


In [54]:
query = """MATCH (s:Species)
WHERE s.taxon_class IN [
  'Clitellata',
  'Diplopoda',
  'Polychaeta',
  'Malacostraca',
  'Entognatha',
  'Chromadorea',
  'Polyplacophora',
  'Agaricomycetes',
  'Saccharomycetes'
]
RETURN s.scientific_name, s.common_name, s.taxon_class, s.taxon_phylum
ORDER BY s.scientific_name, s.taxon_class, s.taxon_phylum;"""

result = gds.run_cypher(query)
print(result)

              s.scientific_name                   s.common_name  \
0               Alitta williami                            None   
1              Anurida maritima             Seashore Springtail   
2                 Armadillidium                        Pillbugs   
3      Armadillidium arcangelii      Arcangeli's Pill Woodlouse   
4         Armadillidium vulgare           Common Pill Woodlouse   
5           Callinectes sapidus              Atlantic Blue Crab   
6              Cancer productus                   Red Rock Crab   
7               Carcinus maenas             European Green Crab   
8           Caulorhiza umbonata                  Redwood Rooter   
9          Centrobolus richardi      Richards Bay Red Millipede   
10        Chicobolus spinigerus         Florida Ivory Millipede   
11          Coprinopsis lagopus              hare's foot inkcap   
12        Cryptochiton stelleri                  Gumboot Chiton   
13              Emerita analoga               Pacific Sand Cra

In [17]:
result.to_csv("verification/decomposers_animals_total.csv", index=False)

#### Decomposers that are obsered as feeding in the dataset

In [15]:
query ="""MATCH (a:Species)-[:eaten_by]->(s:Species)
WHERE s.taxon_class IN [
  'Clitellata',
  'Diplopoda',
  'Polychaeta',
  'Malacostraca',
  'Entognatha',
  'Chromadorea',
  'Polyplacophora',
  'Agaricomycetes',
  'Saccharomycetes'
]
RETURN s.scientific_name, s.common_name, s.taxon_class, s.taxon_phylum
ORDER BY s.scientific_name, s.taxon_class, s.taxon_phylum;
"""

result = gds.run_cypher(query)
print(result)

         s.scientific_name         s.common_name   s.taxon_class  \
0         Anurida maritima   Seashore Springtail      Entognatha   
1  Hemigrapsus oregonensis     Yellow Shore Crab    Malacostraca   
2       Hirudo michaelseni  Common African Leech      Clitellata   
3   Phylloporia amplectens       Pawpaw Polypore  Agaricomycetes   
4   Phylloporia amplectens       Pawpaw Polypore  Agaricomycetes   
5      Ptenothrix maculosa                  None      Entognatha   

  s.taxon_phylum  
0     Arthropoda  
1     Arthropoda  
2       Annelida  
3  Basidiomycota  
4  Basidiomycota  
5     Arthropoda  


In [16]:
result.to_csv("verification/decomposers_animals_feeding.csv", index=False)

In [56]:
query ="""MATCH (a:Species)-[:eaten_by]->(s:Species)
WHERE s.taxon_class IN [
  'Clitellata',
  'Diplopoda',
  'Polychaeta',
  'Malacostraca',
  'Entognatha',
  'Chromadorea',
  'Polyplacophora'
]
RETURN DISTINCT s.scientific_name, s.common_name, s.taxon_class, s.taxon_phylum
ORDER BY s.scientific_name, s.taxon_class, s.taxon_phylum;
"""
result = gds.run_cypher(query)
print(result)


         s.scientific_name         s.common_name s.taxon_class s.taxon_phylum
0         Anurida maritima   Seashore Springtail    Entognatha     Arthropoda
1  Hemigrapsus oregonensis     Yellow Shore Crab  Malacostraca     Arthropoda
2       Hirudo michaelseni  Common African Leech    Clitellata       Annelida
3      Ptenothrix maculosa                  None    Entognatha     Arthropoda


### Herbivores

In [ ]:
query = """MATCH (plants:Species)-[:eaten_by]->(herbivore:Species)
WHERE plants.taxon_kingdom IN ['Plantae','Chromista']
  AND NOT EXISTS {
    MATCH (other:Species)-[:eaten_by]->(herbivore)
    WHERE other.taxon_kingdom IN ['Animalia','Fungi']
  }
RETURN DISTINCT herbivore.scientific_name AS herbivore_name,
       herbivore.common_name AS common_name,
       collect(DISTINCT plants.taxon_kingdom) AS eaten_kingdoms
ORDER BY herbivore_name;
"""

result = gds.run_cypher(query)
print(result)

                herbivore_name                 common_name eaten_kingdoms
0            Abantis paradisea            Paradise Skipper      [Plantae]
1        Abisares viridipennis  Notched Shield Grasshopper      [Plantae]
2              Acalitus mallyi       Mispel Leaf Gall Mite      [Plantae]
3              Aceria camdeboo                        None      [Plantae]
4              Aceria lantanae     Lantana Flower Gallmite      [Plantae]
..                         ...                         ...            ...
463  Zonocerus elegans elegans                        None      [Plantae]
464     Zonotrichia albicollis      White-throated Sparrow      [Plantae]
465           Zosterops virens        Green Cape White-eye      [Plantae]
466    Zosterops virens virens        Green Cape White-eye      [Plantae]
467       Zygaena filipendulae             Six-spot Burnet      [Plantae]

[468 rows x 3 columns]


### Bridges

In [10]:
query = """CALL gds.bridges.stream('foodweb_undirected')
YIELD from, to, remainingSizes
WHERE NOT ANY(x IN remainingSizes WHERE x <10)
RETURN DISTINCT 
  gds.util.asNode(from).scientific_name AS fromName, 
  gds.util.asNode(to).scientific_name AS toName, 
  remainingSizes
ORDER BY size(remainingSizes) DESC;"""

result = gds.run_cypher(query)
print(result)

                   fromName                         toName remainingSizes
0         Crinifer concolor             Adansonia digitata      [819, 10]
1              Ficus bussei              Crinifer concolor      [814, 15]
2      Crocodylus niloticus  Connochaetes taurinus mearnsi      [811, 18]
3        Turdus migratorius              Prunus virginiana      [818, 11]
4             Pica hudsonia              Rattus norvegicus      [816, 13]
5          Sturnus vulgaris              Falco columbarius      [819, 10]
6          Podarcis muralis               Sturnus vulgaris      [818, 11]
7         Vespula germanica               Podarcis muralis      [814, 15]
8        Tringa semipalmata                Emerita analoga      [819, 10]
9          Falco peregrinus             Tringa semipalmata      [818, 11]
10         Colaptes auratus               Falco peregrinus      [815, 14]
11           Astur cooperii               Colaptes auratus      [813, 16]
12   Leucocelis aeneicollis     Syzygi

#### Check if bridges have variable path length

In [12]:
query = """CALL gds.bridges.stream('foodweb_undirected')
YIELD from, to, remainingSizes
WHERE NOT ANY(x IN remainingSizes WHERE x <10)
WITH gds.util.asNode(from) AS startNode, gds.util.asNode(to) AS endNode, remainingSizes
CALL gds.shortestPath.dijkstra.stream('foodweb_undirected', {
  sourceNode: startNode,
  targetNode: endNode
})
YIELD path
RETURN 
  startNode.scientific_name AS fromName,
  endNode.scientific_name AS toName,
  remainingSizes,
  length(path) AS shortestPathLength,
  size(nodes(path)) - 1 AS hopsBetweenNodes
ORDER BY shortestPathLength DESC;"""

result = gds.run_cypher(query)
print(result)

                   fromName                         toName remainingSizes  \
0         Crinifer concolor             Adansonia digitata      [819, 10]   
1              Ficus bussei              Crinifer concolor      [814, 15]   
2      Crocodylus niloticus  Connochaetes taurinus mearnsi      [811, 18]   
3        Turdus migratorius              Prunus virginiana      [818, 11]   
4             Pica hudsonia              Rattus norvegicus      [816, 13]   
5          Sturnus vulgaris              Falco columbarius      [819, 10]   
6          Podarcis muralis               Sturnus vulgaris      [818, 11]   
7         Vespula germanica               Podarcis muralis      [814, 15]   
8        Tringa semipalmata                Emerita analoga      [819, 10]   
9          Falco peregrinus             Tringa semipalmata      [818, 11]   
10         Colaptes auratus               Falco peregrinus      [815, 14]   
11           Astur cooperii               Colaptes auratus      [813, 16]   

### Bridge Species 

In [15]:
query = """CALL gds.articulationPoints.stream('foodweb_undirected')
YIELD nodeId, resultingComponents
RETURN gds.util.asNode(nodeId).scientific_name AS name, resultingComponents.count as number_of_resulting_componenets
ORDER BY resultingComponents.count  DESC"""

result = gds.run_cypher(query)
print(result)

                                  name  number_of_resulting_componenets
0                       Apis mellifera                               27
1                       Ardea herodias                               14
2                 Diospyros virginiana                               11
3                 Toxomerus marginatus                               10
4                         Ficus burkei                                9
..                                 ...                              ...
416                Cyanomitra olivacea                                2
417             Cymatogaster aggregata                                2
418  Cyperus polystachyos polystachyos                                2
419                    Cyperus solidus                                2
420                   Dais cotinifolia                                2

[421 rows x 2 columns]


In [16]:
query = """CALL gds.articulationPoints.stream('foodweb_undirected')
YIELD nodeId, resultingComponents
WHERE resultingComponents.count >= 10
RETURN gds.util.asNode(nodeId).scientific_name AS name,  gds.util.asNode(nodeId).common_name as common_name, resultingComponents.count as number_of_resulting_componenets
ORDER BY resultingComponents.count  DESC"""

result = gds.run_cypher(query)
print(result)

                   name            common_name  \
0        Apis mellifera      Western Honey Bee   
1        Ardea herodias       Great Blue Heron   
2  Diospyros virginiana     American persimmon   
3  Toxomerus marginatus  Margined Calligrapher   

   number_of_resulting_componenets  
0                               27  
1                               14  
2                               11  
3                               10  


In [20]:
query = """CALL gds.articulationPoints.stream('foodweb_undirected')
YIELD nodeId, resultingComponents
WHERE resultingComponents.count > 5
RETURN gds.util.asNode(nodeId).scientific_name AS name,  gds.util.asNode(nodeId).common_name as common_name, resultingComponents.count as number_of_resulting_componenets
ORDER BY resultingComponents.count  DESC"""

result = gds.run_cypher(query)
print(result)

                                name                common_name  \
0                     Apis mellifera          Western Honey Bee   
1                     Ardea herodias           Great Blue Heron   
2               Diospyros virginiana         American persimmon   
3               Toxomerus marginatus      Margined Calligrapher   
4                       Ficus burkei            Common Wild Fig   
5                     Justicia flava            Yellow Justicia   
6   Larus glaucescens × occidentalis               Olympic Gull   
7                 Loxodonta africana   African Savanna Elephant   
8                     Astur cooperii              Cooper's Hawk   
9                    Cirsium arvense           creeping thistle   
10                    Dione vanillae            Gulf Fritillary   
11          Erythemis simplicicollis           Eastern Pondhawk   
12          Haliaeetus leucocephalus                 Bald Eagle   
13                 Larus glaucescens       Glaucous-winged Gul